# 原始产能利用率的固定信息增量检验

先检验行业信息是否改善未来20个交易日收益和下行波动预测。价格/发布日历对照与增加三个原始容量数值的模型各三次年度拟合；固定Ridge=10，不搜索阈值或窗口。标签重叠用20日区块评估，终端未成熟标签保留未知。此步骤不签发净超额或晋升。

In [1]:
import os,sys,json
from pathlib import Path
root=Path.cwd()
while not (root/"SPEC.md").is_file():root=root.parent
sys.path.insert(0,str(root));os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
card=root/"research/fed_capacity_information_20261013"
plan=json.loads((card/"plan.json").read_text())
from src.workflow.fed_capacity_information import run_capacity_information
summary=run_capacity_information(root=root,card=card,plan=plan)
summary

{'experiment': 'fed_capacity_information_20261013',
 'plan_sha256': '03f9fd7825dbb4c23fd2c295ddb5136c5de0b7c4db9cbaecbbf33aee84f85db7',
 'decision': 'not_promoted_capacity_information_gate_failed',
 'information_gate_pass': False,
 'objective_achieved': False,
 'goal_remains_active': True,
 'new_model_fits': 6,
 'backtests': 0,
 'training_audit': [{'fold': 'development_2021',
   'family': 'price_calendar',
   'training_rows': 538,
   'cutoff': '2021-01-01 00:00:00',
   'latest_label_available_at': '2020-12-31 09:49:00',
   'columns': ['soxx_ret20',
    'soxx_vol20',
    'soxx_drawdown60',
    'capacity_known',
    'release_age_days'],
   'targets': ['market_return_h20', 'log_downside_variance_h20'],
   'alpha': 10.0,
   'daily_labels_overlap': True},
  {'fold': 'development_2021',
   'family': 'capacity',
   'training_rows': 538,
   'cutoff': '2021-01-01 00:00:00',
   'latest_label_available_at': '2020-12-31 09:49:00',
   'columns': ['soxx_ret20',
    'soxx_vol20',
    'soxx_drawdown60

In [2]:
# 只修正评价时点：复用6个冻结模型与预测，不重新拟合。
import qlib, numpy as np, hashlib
import pandas as pd
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
from src.workflow.fed_capacity_information import evaluation_mask, _bootstrap
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
qlib.init(provider_uri=summary["provider_uri"],region="us",kernels=1)
names=["inputs/grouped","analysis/predictions","analysis/metrics","analysis/increment","analysis/summary"]+[f"models/{fold['name']}_{family}" for fold in plan["folds"] for family in ("price_calendar","capacity")]
objects,read_audit=load_verified_recorder_objects(card,expected_recorder_id=summary["recorder_id"],artifacts=names)
predictions=objects["analysis/predictions"]
rows=[];increments=[];losses={};excluded={}
for fold in plan["folds"]:
    per_family={}
    for family in ("price_calendar","capacity"):
        entry=predictions[(fold["name"],family)]
        truth,predicted=entry["truth"],entry["prediction"]
        meta=objects["inputs/grouped"]["meta"].reindex(truth.index)
        valid=evaluation_mask(truth,meta,fold,plan["as_of"])
        excluded[fold["name"]]=int((truth.notna().all(axis=1)&~valid).sum())
        error=predicted.loc[valid]-truth.loc[valid]
        model=objects[f"models/{fold['name']}_{family}"]
        per_family[family]=((error.to_numpy()/model.y_scaler.scale_)**2).mean(axis=1)
        entry["label_mature"]=valid
        for target in plan["targets"]:
            rows.append({"fold":fold["name"],"family":family,"target":target,"observations":int(valid.sum()),"rmse":float(np.sqrt(np.mean(error[target]**2))),"bias":float(error[target].mean()),"rank_ic":float(predicted.loc[valid,target].corr(truth.loc[valid,target],method="spearman"))})
    difference=per_family["price_calendar"]-per_family["capacity"]
    losses[fold["name"]]=difference
    increments.append({"fold":fold["name"],**_bootstrap(difference,plan)})
pooled=_bootstrap(np.concatenate([losses[f["name"]] for f in plan["folds"] if f["name"].startswith("development")]),plan)
increments.append({"fold":"pooled_development",**pooled})
passed=all(x["mean_joint_normalized_loss_improvement"]>0 for x in increments if x["fold"].startswith("development")) and pooled["lower_bound"]>0
summary.update(information_gate_pass=bool(passed),decision="information_gate_pass_needs_portfolio_experiment" if passed else "not_promoted_capacity_information_gate_failed",information_increment=increments,evaluation_maturity_correction={"development":"labels must mature before the next calendar year","outer":"labels must mature before as_of16:01","excluded_late_labels":excluded,"new_fits":0,"prediction_values_changed":False})
metrics=pd.DataFrame(rows);increment=pd.DataFrame(increments)
with R.uri_context(summary["recorder_uri"]):
    recorder=R.get_recorder(experiment_id=summary["experiment_id"],recorder_id=summary["recorder_id"])
    recorder.save_objects(artifact_path="analysis",metrics_pre_maturity_correction=objects["analysis/metrics"],increment_pre_maturity_correction=objects["analysis/increment"],summary_pre_maturity_correction=objects["analysis/summary"])
    saved=dict(summary);saved.pop("nas_evidence",None)
    recorder.save_objects(artifact_path="analysis",metrics=metrics,increment=increment,predictions=predictions,paired_losses=losses,summary=saved)
    recorder.save_objects(artifact_path="sources",evaluation_maturity_corrected_code=(root/"src/workflow/fed_capacity_information.py").read_text())
    artifacts=Path(summary["recorder_local_dir"])/"artifacts"
    recorder.log_metrics(registered_artifact_count=len([x for x in artifacts.rglob("*") if x.is_file()]))
metrics.to_csv(card/"forecast_metrics.csv",index=False);increment.to_csv(card/"information_increment.csv",index=False)
atomic_json({"recorder_id":summary["recorder_id"],"entries":[{"path":str(x.relative_to(card)),"sha256":hashlib.sha256(x.read_bytes()).hexdigest(),"bytes":x.stat().st_size} for x in sorted(artifacts.rglob("*")) if x.is_file()]},card/"verification_manifest.json")
summary["nas_evidence"]=archive_nas_recorder(card/"nas_binding.json",workspace_root=root,card=card,recorder_id=summary["recorder_id"],experiment_id=summary["experiment_id"])
atomic_json(summary,card/"summary.json")
text=f"# 原始产能利用率信息增量结论\n\n已执行6次固定线性拟合，新增组合回测0次。决定：`{summary['decision']}`。\n\n|窗口|联合标准化误差改善|区块下界|区块上界|\n|---|---:|---:|---:|\n"
for row in increments:text+=f"|{row['fold']}|{row['mean_joint_normalized_loss_improvement']:.4f}|{row['lower_bound']:.4f}|{row['upper_bound']:.4f}|\n"
text+=f"\n正数代表相对价格/发布日历对照的预测误差下降。20日收益和log下行方差按训练期尺度等权评价，单目标结果保留。开发期评价只使用当年末前已成熟标签，排除跨入下一年标签：{excluded}；修正只重算指标，没有重新训练或改变预测。\n\n"
text+="旧行业口径不拼接；只用当时公开版本和下一交易日可用时点。20日区块不能消除全部研究选择偏差；历史仍是复用诊断。信息门通过还需净收益、风险和独立前向，未通过则结束此固定来源/目标/线性模型路线。目标继续active，不将来源资格或误差改善当作稳定超额。\n"
(card/"conclusion.md").write_text(text)
summary["information_increment"]


[{'fold': 'development_2021',
  'observations': 231,
  'mean_joint_normalized_loss_improvement': -0.604182284407184,
  'lower_bound': -1.0611681191025646,
  'upper_bound': -0.16482455989183672,
  'method': 'paired stationary bootstrap of normalized loss difference',
  'block_length': 20,
  'does_not_correct_all_previous_research_selection': True},
 {'fold': 'development_2022',
  'observations': 230,
  'mean_joint_normalized_loss_improvement': -0.05784201352585806,
  'lower_bound': -0.15796690874852798,
  'upper_bound': 0.03977927652726658,
  'method': 'paired stationary bootstrap of normalized loss difference',
  'block_length': 20,
  'does_not_correct_all_previous_research_selection': True},
 {'fold': 'reused_outer',
  'observations': 925,
  'mean_joint_normalized_loss_improvement': -0.031205334641931273,
  'lower_bound': -0.14746463712401772,
  'upper_bound': 0.07704957908805819,
  'method': 'paired stationary bootstrap of normalized loss difference',
  'block_length': 20,
  'does_no

In [3]:
import pandas as pd
from IPython.display import Markdown,display
display(pd.read_csv(card/"forecast_metrics.csv"))
display(Markdown((card/"conclusion.md").read_text()))

,fold,family,target,observations,rmse,bias,rank_ic
0,development_2021,price_calendar,market_return_h20,231,0.058565,0.000743,0.224104
1,development_2021,price_calendar,log_downside_variance_h20,231,0.855789,0.137718,-0.141020
2,development_2021,capacity,market_return_h20,231,0.064667,-0.027844,0.270143
3,development_2021,capacity,log_downside_variance_h20,231,1.265260,0.806712,-0.634118
4,development_2022,price_calendar,market_return_h20,230,0.129130,0.087367,0.514442
5,development_2022,price_calendar,log_downside_variance_h20,230,0.875907,-0.676612,-0.353280
6,development_2022,capacity,market_return_h20,230,0.130540,0.089709,0.445819
7,development_2022,capacity,log_downside_variance_h20,230,0.899588,-0.705595,-0.407649
8,reused_outer,price_calendar,market_return_h20,925,0.101793,-0.020112,-0.075319
9,reused_outer,price_calendar,log_downside_variance_h20,925,0.828152,-0.075299,0.292574


# 原始产能利用率信息增量结论

已执行6次固定线性拟合，新增组合回测0次。决定：`not_promoted_capacity_information_gate_failed`。

|窗口|联合标准化误差改善|区块下界|区块上界|
|---|---:|---:|---:|
|development_2021|-0.6042|-1.0612|-0.1648|
|development_2022|-0.0578|-0.1580|0.0398|
|reused_outer|-0.0312|-0.1475|0.0770|
|pooled_development|-0.3316|-0.6346|-0.0822|

正数代表相对价格/发布日历对照的预测误差下降。20日收益和log下行方差按训练期尺度等权评价，单目标结果保留。开发期评价只使用当年末前已成熟标签，排除跨入下一年标签：{'development_2021': 21, 'development_2022': 21, 'reused_outer': 0}；修正只重算指标，没有重新训练或改变预测。

旧行业口径不拼接；只用当时公开版本和下一交易日可用时点。20日区块不能消除全部研究选择偏差；历史仍是复用诊断。信息门通过还需净收益、风险和独立前向，未通过则结束此固定来源/目标/线性模型路线。目标继续active，不将来源资格或误差改善当作稳定超额。


# 原始产能利用率信息增量结论

已执行6次固定线性拟合，新增组合回测0次。决定：`not_promoted_capacity_information_gate_failed`。

|窗口|联合标准化误差改善|区块下界|区块上界|
|---|---:|---:|---:|
|development_2021|-0.6042|-1.0612|-0.1648|
|development_2022|-0.0578|-0.1580|0.0398|
|reused_outer|-0.0312|-0.1475|0.0770|
|pooled_development|-0.3316|-0.6346|-0.0822|

正数代表相对价格/发布日历对照的预测误差下降。20日收益和log下行方差按训练期尺度等权评价，单目标结果保留。开发期评价只使用当年末前已成熟标签，排除跨入下一年标签：{'development_2021': 21, 'development_2022': 21, 'reused_outer': 0}；修正只重算指标，没有重新训练或改变预测。

旧行业口径不拼接；只用当时公开版本和下一交易日可用时点。20日区块不能消除全部研究选择偏差；历史仍是复用诊断。信息门通过还需净收益、风险和独立前向，未通过则结束此固定来源/目标/线性模型路线。目标继续active，不将来源资格或误差改善当作稳定超额。
